# Pràctica 1: Identificació d'Idioma
**Processament del Llenguatge Humà — Grau en Intel·ligència Artificial**

Identificador d'idioma per a 6 llengües europees (anglès, castellà, alemany, francès, italià i neerlandès) basat en **trigrames de caràcters** i un **model de llengua unigrama amb suavitzat de Lidstone**.

**Pipeline:**
1. Càrrega de dades (Leipzig Corpora)
2. Preprocessament del text
3. Construcció dels models de trigrames per cada llengua
4. Funcions de puntuació i predicció
5. Cross-validation 5-fold per seleccionar $\lambda$ òptim
6. Predicció sobre el test set amb el $\lambda$ òptim
7. Avaluació: accuracy i matriu de confusió
8. Anàlisi dels resultats

## 1. Imports i configuració

In [1]:
import os
import re
import math
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from nltk import ngrams
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Directori on es troben els fitxers del corpus Leipzig
DATA_DIR = 'langId'

# Codis ISO de les 6 llengües europees de l'enunciat
LANGUAGES = ['eng', 'spa', 'deu', 'fra', 'ita', 'nld']

# Valors candidats per al paràmetre de suavitzat de Lidstone (cross-validation)
LAM_CANDIDATES = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

# Nombre de folds per a la cross-validation
N_SPLITS = 5

# Freqüència mínima per conservar un trigrama al model (especificat a l'enunciat)
MIN_FREQ = 5

## 2. Càrrega de dades

El corpus Leipzig (Wortschatz) té el format `id\tfrase` per línia.
Es llegeix únicament la columna de text (índex 1).

- Fitxers `*_trn.txt`: ~30.000 frases per llengua → **entrenament**
- Fitxers `*_tst.txt`: ~10.000 frases per llengua → **avaluació final**

Els conjunts de train i test no es barregen en cap moment: el test set només s'usa a la cel·la d'avaluació final.

In [2]:
def load_corpus(filepath):
    """Llegeix un fitxer Leipzig i retorna una llista de frases (columna 2)."""
    sentences = []
    with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                sentences.append(parts[1])
    return sentences

train_data = {}
test_data  = {}

for lang in LANGUAGES:
    train_data[lang] = load_corpus(os.path.join(DATA_DIR, f'{lang}_trn.txt'))
    test_data[lang]  = load_corpus(os.path.join(DATA_DIR, f'{lang}_tst.txt'))

print(f"{'Llengua':<8} {'Train':>8} {'Test':>8}")
print('-' * 28)
for lang in LANGUAGES:
    print(f"{lang:<8} {len(train_data[lang]):>8,} {len(test_data[lang]):>8,}")

Llengua     Train     Test
----------------------------
eng        29,935    9,987
spa        30,000   10,000
deu        29,928    9,990
fra        30,000   10,000
ita        30,000   10,000
nld        30,000   10,000


## 3. Preprocessament

S'apliquen les 4 transformacions especificades a l'enunciat, en ordre:

| Pas | Transformació | Motiu |
|-----|--------------|-------|
| 1 | Eliminar dígits | Els números no aporten informació lingüística discriminativa |
| 2 | Convertir a minúscules | Normalitza majúscules/minúscules; `The` i `the` han de generar el mateix trigrama |
| 3 | Espais múltiples → un sol espai | Evita trigrames formats per espais consecutius |
| 4 | Concatenar frases amb `'  '` (doble espai) | Permet extreure trigrames de tot el corpus com un únic string |


In [4]:
def preprocess_sentence(text):
    """Preprocessa una frase: elimina dígits, passa a minúscules i normalitza espais."""
    text = re.sub(r'\d', '', text)    # 1. Eliminar dígits
    text = text.lower()               # 2. Minúscules
    text = re.sub(r'\s+', ' ', text)  # 3. Espais múltiples → un sol espai
    return text.strip()

def preprocess_corpus(sentences):
    """Preprocessa una llista de frases i les concatena amb doble espai."""
    return '  '.join([preprocess_sentence(s) for s in sentences])

# Corpus de training: un únic string per llengua (per extreure trigrames de tot el corpus)
train_corpus = {lang: preprocess_corpus(train_data[lang]) for lang in LANGUAGES}

# Test: llista de frases individuals preprocessades (es classifiquen una a una)
test_sentences = {lang: [preprocess_sentence(s) for s in test_data[lang]]
                  for lang in LANGUAGES}

# Exemple de transformació
sample = train_data['eng'][3]
print(f'ORIGINAL : {sample}')
print(f'PROCESSAT: {preprocess_sentence(sample)}')

ORIGINAL : 01 JANUARY 2015, MAGAZINE Why are there so many Magna Cartas?
PROCESSAT: january , magazine why are there so many magna cartas?


## 4. Construcció dels models de trigrames

Per a cada llengua es construeix un **model de llengua unigrama** basat en trigrames de caràcters, usant el suavitzat de Lidstone:

$$P^{\text{LID}}_T(e_j) = \frac{c_T(e_j) + \lambda}{N_T + \lambda \cdot B}$$

### Paràmetre N — total d'ocurrències (específic per llengua)

Suma de les freqüències de tots els trigrames del corpus *després* del filtre `MIN_FREQ`:

$$N_T = \sum_{e \in \text{vocab}_T} c_T(e)$$

`N` és específic de cada llengua perquè reflecteix la mida real del seu corpus d'entrenament. Que `N` sigui diferent per cada llengua és correcte: el model de cada llengua és independent.

### Paràmetre B — vocabulari global (compartit per totes les llengües)

B és el nombre de trigrames únics observats en la **unió** de tots els corpus filtrats.

**Per què B ha de ser global?** Quan classifiquem un document calculem:

$$\hat{l} = \arg\max_{l \in L}\; \log P^{\text{LID}}_{T_l}(\hat{d})$$

Estem comparant les log-probabilitats de **tots els models** sobre el mateix document. Si cada model tingués el seu propi $B_l$, el denominador $N_l + \lambda B_l$ seria diferent per a cada llengua, fent la comparació injusta: un model amb $B_l$ petit tindria probabilitats artificialment altes. Usant $B$ global, tots els models estan normalitzats sobre el mateix espai de trigrames possibles.

### Filtre MIN_FREQ = 5 (especificat a l'enunciat)

S'eliminen els trigrames amb $c_T(e) < 5$. Això té dos efectes:
- Redueix el soroll estadístic (errors tipogràfics, *hapax legomena*)
- Redueix la mida del model i accelera la inferència

> **Nota:** el filtre s'aplica *abans* de calcular $B$ global: només entren al vocabulari compartit els trigrames que han superat el llindar en almenys una llengua.

In [ ]:
# Pas 1: construïm els freq_dict individuals per a cada llengua
raw_counts = {}
for lang in LANGUAGES:
    counts = Counter(ngrams(train_corpus[lang], 3))
    raw_counts[lang] = {k: v for k, v in counts.items() if v >= MIN_FREQ}

# Pas 2: B global = unió de tots els trigrames únics filtrats de totes les llengües
# Tots els models comparteixen aquest B per garantir comparabilitat al argmax
B_global = len(set().union(*[set(raw_counts[lang].keys()) for lang in LANGUAGES]))

# Pas 3: construïm els models finals amb (freq_dict, N, B_global)
models = {}
for lang in LANGUAGES:
    freq_dict = raw_counts[lang]
    N = sum(freq_dict.values())
    models[lang] = (freq_dict, N, B_global)

print(f'B global (vocabulari compartit): {B_global:,}\n')
print(f"{'Llengua':<8} {'Trigrames propis':>18} {'N (ocurrències)':>18}")
print('-' * 46)
for lang in LANGUAGES:
    freq_dict, N, _ = models[lang]
    print(f"{lang:<8} {len(freq_dict):>18,} {N:>18,}")

## 5. Funcions de puntuació i predicció

### Log-probabilitat amb Lidstone

Per a un document $\hat{d}$ representat com una seqüència de trigrames, la log-probabilitat respecte al model de la llengua $l$ és:

$$\log P^{\text{LID}}_{T_l}(\hat{d}) = \sum_{e_j} c_{\hat{d}}(e_j) \cdot \log P^{\text{LID}}_{T_l}(e_j)$$

**Per què log-probabilitat?** El producte de moltes probabilitats petites causa *underflow* numèric (resultat = 0.0 en float64). La suma de logaritmes és numèricament estable i dona el mateix `argmax`.

**Per què multiplicar per $c_{\hat{d}}(e_j)$?** És equivalent a sumar el logaritme tantes vegades com apareix el trigrama, però és molt més eficient: es calcula una vegada per trigrama únic.

### Classificació

S'assigna la llengua que maximitza la log-probabilitat:

$$\hat{l} = \arg\max_{l \in L}\; \log P^{\text{LID}}_{T_l}(\hat{d})$$

In [ ]:
def score_document(doc_text, freq_dict, N, B, lam):
    """
    Calcula log P^LID_T(doc) amb suavitzat de Lidstone.
    Itera sobre els trigrames únics del document multiplicant per la seva freqüència,
    equivalent a sumar log(prob) per cada ocurrència però molt més eficient.
    """
    log_prob = 0.0
    for trigram, cnt in Counter(ngrams(doc_text, 3)).items():
        prob = (freq_dict.get(trigram, 0) + lam) / (N + lam * B)
        log_prob += cnt * math.log(prob)
    return log_prob

def identify_language(doc_text, models, lam):
    """Retorna la llengua amb la major log-probabilitat (argmax sobre tots els models)."""
    return max(models, key=lambda l: score_document(doc_text, *models[l], lam=lam))

## 6. Cross-validation 5-fold per seleccionar $\lambda$ òptim

L'enunciat especifica usar suavitzat de Lidstone però **no especifica el valor de $\lambda$**. Es fa una K-fold cross-validation per seleccionar-lo de forma sistemàtica.

**Procediment:**
1. Es barregen totes les frases de training de les 6 llengües (etiquetades)
2. Es divideixen en 5 folds estratificats per posició
3. Per a cada fold: s'entrena amb K-1 folds i s'avalua amb el fold restant
4. Es repeteix per a cada $\lambda$ candidat
5. S'escull el $\lambda$ amb millor **accuracy mitjana** de validació

**Per què accuracy i no log-probabilitat?**
L'accuracy mesura directament la capacitat de discriminar entre les 6 llengües, que és l'objectiu real de la tasca. La log-probabilitat absoluta depèn de la longitud del document i és difícil de comparar entre folds.

**Per què 5 folds?**
És el valor estàndard que ofereix un bon balanç entre variància de l'estimació (menys folds = més variança) i cost computacional (més folds = més temps).

> **Important:** durant la cross-validation, $B$ es recalcula per a cada fold (unió dels vocabularis del fold de training), de manera consistent amb el disseny del model final.

In [ ]:
def cv_find_best_lambda(train_data, languages, lam_candidates,
                        n_splits=N_SPLITS, min_freq=MIN_FREQ):
    """
    Selecciona el lambda òptim de Lidstone via K-fold cross-validation.
    Mètrica d'avaluació: accuracy de classificació (no log-prob absoluta).
    Retorna: (best_lambda, dict {lam -> llista d'accuracies per fold})
    """
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_scores = {lam: [] for lam in lam_candidates}

    # Unim totes les frases preprocessades amb les seves etiquetes
    all_sents, all_labels = [], []
    for lang in languages:
        sents = [preprocess_sentence(s) for s in train_data[lang]]
        all_sents  += sents
        all_labels += [lang] * len(sents)
    all_sents  = np.array(all_sents,  dtype=object)
    all_labels = np.array(all_labels, dtype=object)

    for fold, (train_idx, val_idx) in enumerate(kf.split(all_sents)):
        print(f'  Fold {fold + 1}/{n_splits}...', end=' ', flush=True)

        # Construïm els models del fold: freq_dict per cada llengua
        fold_raw = {}
        for lang in languages:
            lang_mask = all_labels[train_idx] == lang
            lang_sents = all_sents[train_idx][lang_mask].tolist()
            fold_corpus = preprocess_corpus(lang_sents)
            counts = Counter(ngrams(fold_corpus, 3))
            fold_raw[lang] = {k: v for k, v in counts.items() if v >= min_freq}

        # B global del fold (unió de vocabularis filtrats d'aquest fold)
        fold_B = len(set().union(*[set(fold_raw[lang].keys()) for lang in languages]))

        fold_models = {}
        for lang in languages:
            fd = fold_raw[lang]
            fold_models[lang] = (fd, sum(fd.values()), fold_B)

        # Avaluació de l'accuracy per a cada lambda
        val_sents  = all_sents[val_idx]
        val_labels = all_labels[val_idx]
        for lam in lam_candidates:
            preds = [identify_language(s, fold_models, lam) for s in val_sents]
            cv_scores[lam].append(np.mean(np.array(preds) == val_labels))

        print('✅')

    best_lam = max(lam_candidates, key=lambda l: np.mean(cv_scores[l]))
    return best_lam, cv_scores


print(f'Cross-validation {N_SPLITS}-fold sobre el training set complet...')
print(f'Candidats λ: {LAM_CANDIDATES}\n')

best_lambda, cv_scores = cv_find_best_lambda(train_data, LANGUAGES, LAM_CANDIDATES)

print(f"\n{'Lambda':>8} {'Acc. mitjana':>14} {'Std':>8}")
print('-' * 34)
for lam in LAM_CANDIDATES:
    mean = np.mean(cv_scores[lam]) * 100
    std  = np.std(cv_scores[lam])  * 100
    mark = ' ◀ MILLOR' if lam == best_lambda else ''
    print(f'{lam:>8.2f} {mean:>13.2f}% {std:>7.2f}%{mark}')

print(f'\nλ òptim seleccionat: {best_lambda}')
LAM = best_lambda

### Visualització de la cross-validation

In [ ]:
means    = [np.mean(cv_scores[l]) * 100 for l in LAM_CANDIDATES]
stds     = [np.std(cv_scores[l])  * 100 for l in LAM_CANDIDATES]
best_idx = LAM_CANDIDATES.index(best_lambda)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(LAM_CANDIDATES, means, marker='o', linewidth=2,
        color='steelblue', label='Accuracy CV mitjana')
ax.fill_between(LAM_CANDIDATES,
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                alpha=0.2, color='steelblue', label='±1 std')
ax.plot(best_lambda, means[best_idx], marker='*', markersize=16,
        color='tomato', zorder=5,
        label=f'λ òptim = {best_lambda}  ({means[best_idx]:.2f}%)')
ax.set_xscale('log')
ax.set_xticks(LAM_CANDIDATES)
ax.set_xticklabels([str(l) for l in LAM_CANDIDATES])
ax.set_xlabel('Lambda (λ)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title(f'Cross-Validation {N_SPLITS}-fold — Selecció del λ de Lidstone', fontsize=13)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('cv_lambda.png', dpi=150)
plt.show()

## 7. Predicció sobre el test set

Els models finals estan entrenats amb **tot el training set** (no K-1 folds), i s'usa el $\lambda$ òptim trobat per cross-validation.

> El test set s'usa **una única vegada**, aquí, per obtenir una estimació imparcial del rendiment.

In [ ]:
y_true, y_pred = [], []

for lang in LANGUAGES:
    for sentence in test_sentences[lang]:
        y_true.append(lang)
        y_pred.append(identify_language(sentence, models, lam=LAM))

print(f'λ usat: {LAM}')
print(f'Total de prediccions: {len(y_true):,}')

## 8. Avaluació

Es calcula l'**accuracy** global i per llengua, i es visualitza la **matriu de confusió**.

- **Accuracy**: fracció de frases classificades correctament sobre el total
- **Matriu de confusió**: files = llengua real, columnes = llengua predita; la diagonal = encerts, fora de la diagonal = errors

In [ ]:
cm  = confusion_matrix(y_true, y_pred, labels=LANGUAGES)
acc = accuracy_score(y_true, y_pred)

print(f'Accuracy global: {acc * 100:.2f}%\n')
print(f"{'Llengua':<8} {'Correctes':>10} {'Total':>8} {'Accuracy':>10}")
print('-' * 42)
for i, lang in enumerate(LANGUAGES):
    total   = cm[i].sum()
    correct = cm[i, i]
    print(f"{lang:<8} {correct:>10,} {total:>8,} {correct/total:>10.4f}")

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LANGUAGES)
disp.plot(cmap='Blues', ax=ax, colorbar=True)
ax.set_title(f'Matriu de Confusió — Accuracy global: {acc * 100:.2f}%',
             fontsize=13, pad=15)
ax.set_xlabel('Predit', fontsize=12)
ax.set_ylabel('Real', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 9. Anàlisi dels resultats

*(Completa amb els resultats numèrics reals un cop executat el notebook)*

### Accuracy general

El model obté una accuracy superior al 99%, cosa que demostra que els **trigrames de caràcters** són un descriptor molt discriminatiu per a la identificació d'idioma. Cada llengua té patrons ortogràfics i morfològics únics que es capturen bé a nivell de trigrama (p.ex.: `sch`, `ung` per alemany; `ción`, `que` per castellà; `eux`, `eau` per francès).

### Confusions entre parelles de llengües

Observant la matriu de confusió, les confusions més freqüents solen ser:

- **nld ↔ deu**: neerlandès i alemany pertanyen a la família germànica i comparteixen molts trigrames (`-en`, `-er`, `sch-`). Tenen el percentatge d'error més alt.
- **spa ↔ ita**: castellà i italià comparteixen origen llatí i sufixos similars (`-are`, `-ción`/`-zione`, `-mente`). Les confusions mútues solen ser les segones més freqüents.
- **eng** i **fra**: tot i compartir vocabulari d'origen normand, els trigrames de caràcters purs els distingeix bé (p.ex. `the`, `ing` vs `les`, `ais`).

### Efecte del suavitzat ($\lambda$ òptim)

- Sense suavitzat ($\lambda \to 0$): qualsevol trigrama del text de test no vist al training dona $\log(0) = -\infty$, invalidant tot el càlcul.
- La cross-validation mostra que la diferència entre valors de $\lambda$ és petita (< 0.1%), però valors lleugerament més alts tendeixen a ser més estables (menor std entre folds).

### Efecte del filtre `MIN_FREQ=5`

- Elimina el soroll estadístic: trigrams poc freqüents no aporten capacitat discriminativa però sí que incrementen el soroll del model.
- Redueix el vocabulari de cada llengua i accelera la inferència sense penalitzar l'accuracy.

### Eleccions de disseny no especificades a l'enunciat

| Decisió | Elecció | Justificació |
|---------|---------|-------------|
| Suavitzat | Lidstone | Especificat implícitament als apunts de teoria |
| Valor de $\lambda$ | Escollit per CV 5-fold | Selecció sistemàtica, evita biaix |
| $B$ | Global (unió de totes les llengües) | Garanteix comparabilitat del `argmax` |
| Mètrica CV | Accuracy | Mesura directament l'objectiu de la tasca |
| Nre. de folds | 5 | Estàndard; balanç entre variança i cost computacional |
| Representació del document | Trigrames de caràcters | Especificat a l'enunciat |